# Merge datasets

In [1]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely import Polygon

In [2]:
def create_square_polygon(e, n, size):
    """create square polygon of size ^ 2 meteres
    Args:
        e: E coordinates
        n: N coordinates
        size: Length and width of the polygon. Defaults to 100.
    """
    return Polygon([(e, n), (e + size, n), (e + size, n + size), (e, n + size)])

### load data

In [3]:
data_folder = os.path.join("data")

statpop_filename = os.path.join(data_folder, "ag-b-00.03-vz2022statpop", "STATPOP2022.csv")
statpop = pd.read_csv(statpop_filename, sep=';', encoding='utf-8', usecols=["E_KOORD", "N_KOORD", "RELI", "B22BTOT"])

statent_filename = os.path.join(data_folder, "ag-b-00.03-22-STATENT2022", "STATENT_2022.csv")
statent = pd.read_csv(statent_filename, sep=';', encoding='utf-8', usecols=["ERHJAHR", "E_KOORD", "N_KOORD", "RELI", "B08T", "B08EMPT", "B08VZAT"])

LAUSANNE_HULL_WKT = (
    "POLYGON ((6.6973149 46.4991489, 6.5474464 46.508319, 6.5441708 46.5091808, "
    "6.5429015 46.5134131, 6.5361596 46.5588201, 6.5949948 46.5926757, "
    "6.6582251 46.5991119, 6.7121747 46.571259, 6.7180473 46.5236707, "
    "6.7159339 46.5179337, 6.7123702 46.5099251, 6.7077559 46.5021425, "
    "6.7016307 46.4999401, 6.6973149 46.4991489))"
)

lausanne = gpd.GeoDataFrame(geometry=gpd.GeoSeries.from_wkt([LAUSANNE_HULL_WKT]), crs="EPSG:4326")

nodes_filename = os.path.join(data_folder, "lausanne_drive_nodes.gpkg")
nodes = gpd.read_file(nodes_filename)

### transform data

In [4]:
statent["geometry"] = statent.apply(lambda row: create_square_polygon(row["E_KOORD"], row["N_KOORD"], 100), axis=1)
statpop["geometry"] = statpop.apply(lambda row: create_square_polygon(row["E_KOORD"], row["N_KOORD"], 100), axis=1)


statent_gpd = gpd.GeoDataFrame(statent, geometry="geometry", crs="EPSG:2056")
statpop_gpd = gpd.GeoDataFrame(statpop, geometry="geometry", crs="EPSG:2056")

In [5]:
statent_lausanne = gpd.sjoin(statent_gpd, lausanne.to_crs("EPSG:2056"), how="inner", predicate="within")
statpop_lausanne = gpd.sjoin(statpop_gpd, lausanne.to_crs("EPSG:2056"), how="inner", predicate="within")
statent_lausanne.drop(columns=["index_right"], inplace=True)
statpop_lausanne.drop(columns=["index_right"], inplace=True)

### Join into one geodataframe

In [6]:
statpop_statent = statpop_lausanne.merge(statent_lausanne, on=["RELI", "N_KOORD", "E_KOORD", "geometry"], how="outer", suffixes=("_statpop", "_statent"))

In [7]:
# join each row in statpop_statent to the nearest node
statpop_statent_nodes = statpop_statent.sjoin_nearest(nodes.to_crs("EPSG:2056"), how="left", distance_col="distance_to_node")